In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

# Install Dependencies

! pip install -U lightning

# Organize Imports

In [ ]:
from pathlib import Path
import functools

import lightning as pl
from lightning.pytorch.callbacks import ModelCheckpoint
import torch
import torch.nn.functional as F
from transformers import BertModel, BertTokenizer
# from torchtext.vocab import GloVe
import numpy as np
import os

# Orginize Path

In [ ]:
PATH = Path('../../data')
model_path = PATH / 'models' / 'word2vec'
model_path.mkdir(parents=True, exist_ok=True)
MNIST_dir = PATH / 'mnist'
MNIST_dir.mkdir(parents=True, exist_ok=True)

# Initialize Device and Workers

In [ ]:
import os
 
workers = os.cpu_count()
print("Number of CPUs in the system:", workers)

In [ ]:
if torch.cuda.is_available():
    device = 'gpu'  
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

device

# Initialize or Download Vectors

In [ ]:
# Load pre-trained model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model = model.eval()

# Helper Functions

In [ ]:
class WordVec(object):

    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model= model
        self.word_matrix = self._embeddings()

    @torch.inference_mode
    def _embeddings(self):
        return self.model.embeddings.word_embeddings.weight

    def tokenize(self, words):
        # Tokenize the words and get their token IDs
        tokens = {word: self.tokenizer.encode(word, add_special_tokens=False) for word in words}
        assert all(len(ids) == 1 for ids in tokens.values()), "Each word should correspond to a single token."
        
        # Get the token IDs
        token_ids = torch.tensor([ids[0] for ids in tokens.values()])
    
        return token_ids

    @torch.inference_mode
    def encode(self, words):
        token_ids = self.tokenize(words)
        # Get the embeddings from the model's embedding layer
        embeddings = self.model.embeddings.word_embeddings(token_ids)
    
        return embeddings

    def decode(self, embd, k=1):
        # Compute cosine similarities
        cosine_similarities = F.cosine_similarity(embd.unsqueeze(0), self.word_matrix)
    
        # Find the index of the most similar word
        most_similar_idx = torch.topk(cosine_similarities, k=k).indices
    
        # Convert the index back to a word
        predicted_word = self.tokenizer.decode(most_similar_idx)
    
        return predicted_word

    def norm(self, emb):
        return emb / emb.norm()

    def meet(self, *vs):
        return functools.reduce(torch.minimum, vs)

    def join(self, *vs):
        return functools.reduce(torch.maximum, vs)

    def meet_w(self, *ws):
        vs = self.encode(ws)
        r = self.meet(vs)
        d = self.decode(r)

        return d

    def join_w(self, *ws):
        vs = self.encode(ws)
        r = self.join(vs)
        d = self.decode(r)

        return d

    def ge(self, emb):
        res = (self.word_matrix >= emb)
        matchs = torch.where(res.all(dim=1))[0]
        # Convert indices back to tokens
        match_tokens = self.tokenizer.convert_ids_to_tokens(matchs.tolist())
        
        return match_tokens
        

In [ ]:
word_vec = WordVec(tokenizer, model)

# Experiments on Lattices

In [ ]:
# Define the words
words = [
    'king', 
    'man', 
    'woman', 
    'royal', 
    'masculine', 
    'feminine',
]
king, man, woman, royal, masculine, feminine = word_vec.encode(words)

In [ ]:
# Perform the vector arithmetic
res = word_vec.norm(king - man + woman)
word_vec.decode(res)

In [ ]:
res = word_vec.meet(feminine, masculine)
word_vec.decode(res)

In [ ]:
res = word_vec.meet(
    king, 
    man, 
    woman, 
    masculine, 
    feminine
)
# word_vec.decode(res), torch.round(res, decimals=2)

In [ ]:
word_vec.ge(res)

In [ ]:
test = word_vec.join(res, man)
word_vec.decode(test)

In [ ]:
# res = word_vec.join(king, woman)
word_vec.decode(res, k=1)

# Experiment with Vectors

In [ ]:
with torch.no_grad():
    all_word_embeddings = model.embeddings.word_embeddings.weight

In [ ]:
embedding_matrix = all_word_embeddings

In [ ]:
# Define the words
words = ['king', 'man', 'woman', 'royal']

# Tokenize the words and get their token IDs
tokens = {word: tokenizer.encode(word, add_special_tokens=False) for word in words}
assert all(len(ids) == 1 for ids in tokens.values()), "Each word should correspond to a single token."

# Get the token IDs
token_ids = torch.tensor([ids[0] for ids in tokens.values()])

# Get the embeddings from the model's embedding layer
with torch.no_grad():
    embeddings = model.embeddings.word_embeddings(token_ids)

# Unpack the embeddings
king_emb, man_emb, woman_emb, royal_emb = embeddings

# Perform the vector arithmetic
result_vector = king_emb - man_emb + woman_emb
result_vector = result_vector / result_vector.norm()



# Compute cosine similarities
cosine_similarities = F.cosine_similarity(result_vector.unsqueeze(0), all_word_embeddings)

# Find the index of the most similar word
most_similar_idx = torch.topk(cosine_similarities, k=2).indices

# Convert the index back to a word
predicted_word = tokenizer.decode(most_similar_idx)

print(f"Result: {predicted_word}")

In [ ]:
what_emb = torch.maximum(king_emb - man_emb, king_emb)

In [ ]:
res = (embeddings >= king_emb)
torch.where(res.all(dim=1))[0]

In [ ]:
res = (all_word_embeddings >= torch.minimum(king_emb, man_emb))
matchs = torch.where(res.all(dim=1))[0]
# Convert indices back to tokens
match_tokens = tokenizer.convert_ids_to_tokens(matchs.tolist())

# Print the matching tokens
print(f"Tokens with embeddings >= the embedding of '{2}':")
for token in match_tokens:
    print(token)

In [ ]:
# torch.minimum(king_emb, man_emb)

In [ ]:
cosine_similarities

In [ ]:
predicted_words = tokenizer.decode(cosine_similarities)

In [ ]:
all_word_embeddings.shape

In [ ]:
torch.round(man_emb, decimals=2)

In [ ]:
strip_emb = king_emb - man_emb
torch.round(strip_emb, decimals=4)

In [ ]:
strip_emb = strip_emb / strip_emb.norm()
torch.round(strip_emb, decimals=4)

In [ ]:
reference_vector = strip_emb

In [ ]:
# Perform element-wise comparison to check if each component of the embeddings
# is greater than or equal to the corresponding component in the reference vector
comparison = embedding_matrix >= king_emb

# Check if all components meet the condition for each embedding
matching_embeddings = comparison.all(dim=0)
torch.where(matching_embeddings)

In [ ]:
reference_word = 'strip'

In [ ]:
# Get the indices of embeddings that match the condition
matching_indices = torch.where(matching_embeddings)[0]

# Convert indices back to tokens
matching_tokens = tokenizer.convert_ids_to_tokens(matching_indices.tolist())

# Print the matching tokens
print(f"Tokens with embeddings >= the embedding of '{reference_word}':")
for token in matching_tokens:
    print(token)

In [ ]:
import torch
from transformers import BertModel, BertTokenizer

# Load pre-trained model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model.eval()

# Access the embedding matrix
embedding_matrix = model.embeddings.word_embeddings.weight

# Define the reference word
reference_word = 'example'

# Tokenize the reference word to get its token ID
reference_token_id = tokenizer.convert_tokens_to_ids(reference_word)

# Retrieve the embedding vector for the reference word
reference_vector = embedding_matrix[reference_token_id]

# Perform element-wise comparison
comparison = embedding_matrix >= reference_vector

# Check if all components meet the condition for each embedding
matching_embeddings = comparison.all(dim=1)

# Get the indices of embeddings that match the condition
matching_indices = torch.where(matching_embeddings)[0]

# Convert indices back to tokens
matching_tokens = tokenizer.convert_ids_to_tokens(matching_indices.tolist())

# Print the matching tokens
print(f"Tokens with embeddings >= the embedding of '{reference_word}':")
for token in matching_tokens:
    print(token)

In [ ]:
import torch
from transformers import BertModel, BertTokenizer
import torch.nn.functional as F

# Load pre-trained model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model.eval()

# Define the words
words = ['king', 'man', 'woman']

# Tokenize the words and get their token IDs
tokens = {word: tokenizer.encode(word, add_special_tokens=False) for word in words}
assert all(len(ids) == 1 for ids in tokens.values()), "Each word should correspond to a single token."

# Get the token IDs
token_ids = torch.tensor([ids[0] for ids in tokens.values()])

# Get the embeddings from the model's embedding layer
with torch.no_grad():
    embeddings = model.embeddings.word_embeddings(token_ids)

# Unpack the embeddings
king_emb, man_emb, woman_emb = embeddings

# Perform the vector arithmetic
result_vector = king_emb - man_emb + woman_emb
result_vector = result_vector / result_vector.norm()

# Get all word embeddings from the model
with torch.no_grad():
    all_word_embeddings = model.embeddings.word_embeddings.weight

# Compute cosine similarities
cosine_similarities = F.cosine_similarity(result_vector.unsqueeze(0), all_word_embeddings)

# Find the index of the most similar word
most_similar_idx = torch.argmax(cosine_similarities).item()

# Convert the index back to a word
predicted_word = tokenizer.decode([most_similar_idx])

print(f"Result: {predicted_word}")